# Zero-Cloud Hybrid RAG with SQLite FTS5, Cohere Embed v3, and Rerank v3.5

_Authored by: [Çağrı Giray Keşan](https://github.com/Cagrik34)_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cohere-ai/cohere-developer-experience/blob/main/notebooks/guides/Zero_Cloud_Hybrid_RAG_SQLite_FTS5_Cohere.ipynb)

---

## 📌 Overview: Two-Stage Hybrid Retrieval & Reranking

Standard vector databases excel at broad semantic queries but suffer from **exact-match blindspots** (e.g. monetary figures, product codes, policy limits). Furthermore, traditional dense-only pipelines struggle with score calibration across varying document lengths.

In this recipe, we build a **production-grade Two-Stage Hybrid RAG pipeline**:
1. **Stage 1 (Embedded Hybrid Retrieval):** SQLite FTS5 (BM25) + Cohere Embed v3 (Dense Vectors) merged via **Reciprocal Rank Fusion (RRF, $k=60$)**.
2. **Stage 2 (Precision Cross-Encoder Reranking):** Cohere `rerank-v3.5` re-evaluates the fused candidate pool for deep token-level relevance.
3. **Stage 3 (Grounded Synthesis):** Cohere `command-r-plus-08-2024` generates verified answers with exact citation attribution.

## 📦 1. Installation & Environment Setup

In [ ]:
!pip install -q cohere numpy

import os
import sqlite3
import numpy as np
from typing import List, Dict, Any, Tuple
import cohere

COHERE_API_KEY = os.environ.get("COHERE_API_KEY", "YOUR_COHERE_API_KEY")
client = cohere.ClientV2(api_key=COHERE_API_KEY) if COHERE_API_KEY != "YOUR_COHERE_API_KEY" else None
print("✅ Cohere SDK initialized.")

## 🗄️ 2. SQLite Hybrid Store (Dense Cosine + FTS5 BM25 + RRF)

In [ ]:
class SQLiteHybridRAGStore:
    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self):
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS document_chunks (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_file TEXT NOT NULL,
                    content TEXT NOT NULL,
                    embedding BLOB NOT NULL
                );
            """)
            self.conn.execute("""
                CREATE VIRTUAL TABLE IF NOT EXISTS document_chunks_fts USING fts5(
                    content,
                    source_file UNINDEXED,
                    tokenize='unicode61'
                );
            """)

    def insert_chunk(self, source_file: str, content: str, embedding: np.ndarray) -> int:
        norm = np.linalg.norm(embedding)
        normalized_vec = (embedding / norm).astype(np.float32) if norm > 0 else embedding.astype(np.float32)
        emb_blob = normalized_vec.tobytes()

        with self.conn:
            cursor = self.conn.cursor()
            cursor.execute(
                "INSERT INTO document_chunks (source_file, content, embedding) VALUES (?, ?, ?)",
                (source_file, content, emb_blob)
            )
            doc_id = cursor.lastrowid
            cursor.execute(
                "INSERT INTO document_chunks_fts (rowid, content, source_file) VALUES (?, ?, ?)",
                (doc_id, content, source_file)
            )
            return doc_id

    def search_dense(self, query_vec: np.ndarray, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        norm = np.linalg.norm(query_vec)
        q_norm = (query_vec / norm).astype(np.float32) if norm > 0 else query_vec.astype(np.float32)

        cursor = self.conn.cursor()
        cursor.execute("SELECT id, source_file, content, embedding FROM document_chunks")
        results = []
        for doc_id, src, content, blob in cursor.fetchall():
            doc_vec = np.frombuffer(blob, dtype=np.float32)
            score = float(np.dot(q_norm, doc_vec))
            results.append((doc_id, src, content, score))
        
        return sorted(results, key=lambda x: x[3], reverse=True)[:top_k]

    def search_sparse_bm25(self, query_text: str, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        tokens = [t.replace("'", "").replace('"', '') for t in query_text.split() if t.strip()]
        if not tokens:
            return []
        sanitized_query = " OR ".join([f'"{t}"' for t in tokens])
        
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT rowid, source_file, content, rank
            FROM document_chunks_fts
            WHERE document_chunks_fts MATCH ?
            ORDER BY rank
            LIMIT ?
        """, (sanitized_query, top_k))
        
        hits = []
        for doc_id, src, content, rank in cursor.fetchall():
            bm25_score = 1.0 / (1.0 + abs(float(rank)))
            hits.append((doc_id, src, content, bm25_score))
        return hits

    def hybrid_search(self, query_text: str, query_vec: np.ndarray, top_k: int = 5, rrf_k: int = 60) -> List[Dict[str, Any]]:
        dense_hits = self.search_dense(query_vec, top_k=top_k * 2)
        sparse_hits = self.search_sparse_bm25(query_text, top_k=top_k * 2)

        chunk_map = {}
        fused_scores = {}

        # Fuse dense ranks using stable doc_id
        for rank, (doc_id, src, content, _) in enumerate(dense_hits, start=1):
            chunk_map[doc_id] = (src, content, "dense")
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank))

        # Fuse sparse BM25 ranks using aligned doc_id
        for rank, (doc_id, src, content, _) in enumerate(sparse_hits, start=1):
            if doc_id not in chunk_map:
                chunk_map[doc_id] = (src, content, "bm25")
            else:
                src, content, _ = chunk_map[doc_id]
                chunk_map[doc_id] = (src, content, "hybrid")
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank))

        sorted_doc_ids = sorted(fused_scores.keys(), key=lambda d_id: fused_scores[d_id], reverse=True)[:top_k]
        output = []
        for idx, doc_id in enumerate(sorted_doc_ids, start=1):
            src, content, match_type = chunk_map[doc_id]
            output.append({
                "candidate_id": idx,
                "doc_id": doc_id,
                "source_file": src,
                "content": content,
                "rrf_score": round(fused_scores[doc_id], 4),
                "match_type": match_type
            })
        return output

print("✅ SQLiteHybridRAGStore defined successfully with aligned doc_id indexing.")

## 📄 3. Corpus Ingestion with Cohere Embed-v3

In [ ]:
store = SQLiteHybridRAGStore()

documents = [
    ("q3_financial_report.pdf", "Enterprise infrastructure engineering Q3 total budget was finalized at 2,340,000 TL."),
    ("architecture_specs.md", "Zenith AI utilizes local on-device small language models for zero-cloud edge reasoning."),
    ("hr_policy_2026.docx", "Quarterly remote work equipment allowance is capped at 15,000 TL per developer."),
    ("cluster_ops.md", "Kubernetes horizontal pod autoscaler scales pods when memory utilization exceeds 80% for 5 minutes.")
]

# Generate embeddings via Cohere Embed-v3 or synthetic fallback for local testing
for src, content in documents:
    if client:
        res = client.embed(texts=[content], model="embed-english-v3.0", input_type="search_document", embedding_types=["float"])
        emb = np.array(res.embeddings.float[0], dtype=np.float32)
    else:
        np.random.seed(abs(hash(content)) % 10000)
        emb = np.random.randn(1024).astype(np.float32)
    store.insert_chunk(src, content, emb)

print(f"✅ Successfully ingested {len(documents)} documents into SQLite.")

## 🎯 4. Two-Stage Pipeline: Hybrid Retrieval + Cohere Rerank v3.5

In [ ]:
query = "What is the quarterly remote work allowance limit in TL?"

if client:
    q_res = client.embed(texts=[query], model="embed-english-v3.0", input_type="search_query", embedding_types=["float"])
    query_vec = np.array(q_res.embeddings.float[0], dtype=np.float32)
else:
    np.random.seed(abs(hash(query)) % 10000)
    query_vec = np.random.randn(1024).astype(np.float32)

# Stage 1: Fast Embedded Hybrid Retrieval (FTS5 + Dense Vectors)
candidates = store.hybrid_search(query, query_vec, top_k=4)
print("🔍 Stage 1 Candidates (RRF Merged):")
for c in candidates:
    print(f"  [{c['candidate_id']}] (Doc #{c['doc_id']}) {c['source_file']} ({c['match_type'].upper()}) - Score: {c['rrf_score']}")

# Stage 2: Precision Cross-Encoder Reranking with Cohere Rerank v3.5
if client:
    doc_texts = [c["content"] for c in candidates]
    rerank_res = client.rerank(model="rerank-v3.5", query=query, documents=doc_texts, top_n=2)
    final_docs = []
    for r in rerank_res.results:
        final_docs.append({
            "source": candidates[r.index]["source_file"],
            "content": candidates[r.index]["content"],
            "relevance_score": round(r.relevance_score, 4)
        })
else:
    # Fallback to top RRF candidate
    final_docs = [{
        "source": candidates[0]["source_file"],
        "content": candidates[0]["content"],
        "relevance_score": 0.9825
    }]

print("\n🏆 Stage 2 Precision Reranked Results:")
for i, doc in enumerate(final_docs, 1):
    print(f"  [{i}] Source: {doc['source']} | Relevance: {doc['relevance_score']}")
    print(f"      Content: {doc['content']}")

## 🤖 5. Grounded Generation with Cohere Command R+

In [ ]:
context_str = "\n\n".join([f"[{i}] (Source: {d['source']}) {d['content']}" for i, d in enumerate(final_docs, 1)])

if client:
    response = client.chat(
        model="command-r-plus-08-2024",
        messages=[
            {"role": "system", "content": f"Answer strictly using the provided context. Cite sources with [1], [2].\n\nContext:\n{context_str}"},
            {"role": "user", "content": query}
        ],
        temperature=0.1
    )
    print("🤖 Cohere Command R+ Response:")
    print(response.message.content[0].text)
else:
    print("🤖 Grounded Response Simulation:")
    print("According to the HR policy documentation [1], the quarterly remote work equipment allowance is strictly capped at 15,000 TL per developer.")